In [ ]:
import sys, pathlib, os

# Ensure repository root on sys.path as required
try:
    sys.path.append(str(pathlib.Path(__file__).resolve().parents[1]))
    repo_root = pathlib.Path(__file__).resolve().parents[1]
except NameError:
    nb_guess = pathlib.Path(os.getcwd()) / "notebooks" / "03_forecast.ipynb"
    repo_root = nb_guess.resolve().parents[1] if nb_guess.exists() else pathlib.Path(os.getcwd()).resolve()
    sys.path.append(str(repo_root))

print(f"Repository root: {repo_root}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.plotting import set_matplotlib_style
from utils.run import RunContext
import utils.config as config

from stages.forecast import run_forecast

# Set plotting style for downstream figures
set_matplotlib_style()

In [ ]:
import json, yaml
from pathlib import Path

cfg_paths = [repo_root / "configs" / "default.yaml", repo_root / "configs" / "forecast.yaml"]
schema_path = repo_root / "configs" / "schema.json"

# Load and merge configs using utils.config if available
cfg = None
try:
    if hasattr(config, "load_config"):
        cfg = config.load_config(paths=[str(p) for p in cfg_paths])
    elif hasattr(config, "load_and_validate"):
        cfg = config.load_and_validate(paths=[str(p) for p in cfg_paths], schema_path=str(schema_path))
except Exception as e:
    print(f"utils.config helper failed: {e}")

if cfg is None:
    # Fallback: YAML read and shallow merge (later keys override earlier)
    merged = {}
    for p in cfg_paths:
        with open(p, "r", encoding="utf-8") as f:
            d = yaml.safe_load(f) or {}
        for k, v in d.items():
            if isinstance(v, dict) and k in merged and isinstance(merged[k], dict):
                merged[k].update(v)
            else:
                merged[k] = v
    cfg = merged

# Validate against schema using utils.config or jsonschema
try:
    if hasattr(config, "validate_config"):
        config.validate_config(cfg, schema_path=str(schema_path))
    else:
        import jsonschema
        with open(schema_path, "r", encoding="utf-8") as f:
            schema = json.load(f)
        jsonschema.validate(instance=cfg, schema=schema)
except Exception as e:
    raise

print("Configuration loaded and validated for forecast stage.")

In [ ]:
from utils.plotting import place_legend_below  # ensure available for downstream plots
import os, random

stage_name = "forecast"

# Start run and stage contexts
run = RunContext.start(cfg)
ctx = run.stage(stage_name)

# Set deterministic seeds
seed = int(cfg.get("seed", 12345))
try:
    from utils.seeds import set_global_seeds
    set_global_seeds(seed)
except Exception:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)

# Optional: structured log at start
try:
    if hasattr(ctx, "log_json"):
        ctx.log_json(level="INFO", stage=stage_name, site_id=None, lineage=None, message="Starting forecast stage", context={"seed": seed})
except Exception:
    pass

In [ ]:
# Run the Forecast stage (PF + MAP + t+1 predictive)
run_forecast(cfg, ctx)

# Close the stage context with provenance
inputs = [str(p) for p in cfg_paths]
notes = "Forecast stage orchestrated via notebooks/03_forecast.ipynb"
try:
    ctx.close(inputs=inputs, notes=notes)
except Exception:
    pass

print("Forecast stage completed.")